# 5. DataOps para Proyectos de Datos Financieros

Hasta ahora, hemos trabajado con notebooks de forma aislada: cada uno se ejecuta de principio a fin y produce un análisis o un modelo. 

Para que se convierta en algo que otras personas puedan **reproducir, auditar, revisar y poner en producción**, hace falta un conjunto de prácticas de ingeniería que se conoce como **DataOps**: la aplicación de principios de desarrollo de software (control de versiones, entornos reproducibles, pruebas automatizadas, integración continua) al trabajo con datos y modelos.

En finanzas esto no es opcional: un modelo de riesgo crediticio o una estrategia de *trading* que no se pueda reproducir exactamente meses después es, en la práctica, no auditable y los reguladores, los comités de riesgo y los propios equipos internos exigen esa trazabilidad.

El camino desde "tengo un notebook en mi computador" hasta "tengo un repositorio con pruebas automatizadas que se validan solas en cada cambio" lo presentamos en los siguientes bloques. 

> **Lectura complementaria:** este cuaderno se apoya en los principios organizacionales descritos en *[Practical DataOps: Delivering Agile Data Science at Scale](01%20Practical%20DataOps%20Delivering%20Agile%20Data%20Science%20At%20Scale.pdf)* (Harvey Atkinson).

## Bloque 1 (3 h) — Entorno local

### 1.1 Python y VS Code: ¿por qué no solo Colab?

Google Colab es un excelente punto de partida para aprender: no requiere instalación y da acceso gratuito a GPU. Pero casi ningún equipo de datos trabaja exclusivamente ahí. Las razones son casi siempre las mismas:

| Necesidad | Colab | Entorno local / servidor propio |
|---|---|---|
| Datos privados (posiciones, clientes, transacciones) | Suben a la infraestructura de Google | Se quedan dentro de la red de la empresa |
| Cumplimiento normativo (auditoría, protección de datos) | Difícil de certificar | Se puede controlar y documentar |
| Control de versiones de librerías | Entorno compartido, cambia sin previo aviso | Ambientes virtuales fijos y reproducibles (Bloque 2) |
| Integración con Git/CI/CD | Limitada | Nativa (VS Code, terminal, GitHub Actions) |
| Cómputo dedicado y costos a escala | Sesión temporal, se desconecta | Servidores propios o en la nube contratados a medida |

La conclusión no es "Colab es malo": es que Colab sirve para explorar y prototipar rápido, mientras que un entorno local (o un servidor dentro de la infraestructura de la empresa) es el que permite construir algo reproducible y seguro. El primer paso para trabajar de esa segunda forma es instalar Python y VS Code.

#### Paso a paso: instalar Python y VS Code

| Paso | Windows | macOS |
|---|---|---|
| 1. Descargar Python | Ir a [python.org/downloads](https://www.python.org/downloads/windows/) y descargar el instalador de la versión más reciente de Python 3 | Ir a [python.org/downloads](https://www.python.org/downloads/macos/) y descargar el instalador `.pkg` — o, si ya tienes [Homebrew](https://brew.sh/), instalar con `brew install python` |
| 2. Instalar | Ejecutar el instalador y **marcar la casilla "Add python.exe to PATH"** antes de darle a "Install Now" — sin esto, la terminal no va a reconocer el comando `python` | Ejecutar el instalador `.pkg` y seguir el asistente (con Homebrew, `python3` queda disponible automáticamente, sin este paso) |
| 3. Verificar la instalación | Abrir *PowerShell* (o *Símbolo del sistema*) y ejecutar `python --version` | Abrir *Terminal* y ejecutar `python3 --version` (en Mac el comando es `python3`, no `python`, salvo que configures un alias) |
| 4. Descargar VS Code | Ir a [code.visualstudio.com](https://code.visualstudio.com/) y descargar el instalador `.exe` | Ir a [code.visualstudio.com](https://code.visualstudio.com/) y descargar el `.zip` |
| 5. Instalar VS Code | Ejecutar el instalador (siguiente → siguiente → instalar) | Descomprimir y arrastrar `Visual Studio Code.app` a la carpeta `Aplicaciones` |
| 6. Abrir la terminal integrada | Dentro de VS Code: `Ctrl` + `` ` `` (acento grave), o menú *Terminal → New Terminal* | Dentro de VS Code: `Cmd` + `` ` ``, o menú *Terminal → New Terminal* |

> Las extensiones de VS Code (Python, Jupyter, GitLens) las instalamos en la sección 1.3 

### 1.2 La terminal: tu primera herramienta de trabajo

La terminal es la forma más directa de decirle a tu computador qué hacer, sin pasar por una interfaz gráfica. La mayoría de los comandos básicos son los mismos en macOS/Linux y en **PowerShell** —la terminal que VS Code abre por defecto en Windows (sección 1.1)—, porque PowerShell incluye alias que reconocen los nombres de Unix. Donde sí hay diferencias, van lado a lado:

| Qué quieres hacer | macOS / Linux (Terminal) | Windows (PowerShell) |
|---|---|---|
| Ver en qué carpeta estás parado | `pwd` | `pwd` |
| Listar el contenido de la carpeta actual | `ls -la` (incluye archivos ocultos) | `ls -Force` (`dir` también funciona) |
| Cambiar de carpeta | `cd proyecto_precios_acciones` | `cd proyecto_precios_acciones` |
| Crear una carpeta (con subcarpetas si hace falta) | `mkdir -p src/datos` | `mkdir src/datos` (PowerShell crea las carpetas intermedias sin necesitar `-p`) |
| Ejecutar un script de Python | `python3 archivo.py` | `python archivo.py` |

> Si en lugar de PowerShell usas el **símbolo del sistema clásico** de Windows (`cmd.exe`, heredado de versiones antiguas): `ls` no existe, usa `dir`; `pwd` tampoco existe, usa `cd` sin argumentos. El resto de la tabla funciona igual.

Vamos a ejecutar varios de estos comandos de verdad, usando `!` al inicio de la celda para enviar el comando directamente a la terminal del sistema (una funcionalidad propia de los notebooks de Jupyter). Este cuaderno corre sobre macOS/Linux, así que las salidas que vas a ver usan esa sintaxis — en tu propia máquina con Windows, sigue la columna de PowerShell de la tabla anterior.

In [1]:
# El signo "!" al inicio de una celda envía el comando a la terminal del sistema,
# no al intérprete de Python
!pwd

/Users/nataliaacevedo/MisCursos/docs/source/analitica_financiera


In [2]:
import tempfile
import pathlib

taller_dir = pathlib.Path(tempfile.mkdtemp(prefix="taller_dataops_"))
%cd {taller_dir}

!mkdir -p bloque1_demo
!ls -la bloque1_demo

/private/var/folders/tf/x2jqms8n4wn8gfljd87ypbl40000gn/T/taller_dataops_85f0v8gv


total 0
drwxr-xr-x  2 nataliaacevedo  staff  64 Sep 18 15:03 .
drwx------  3 nataliaacevedo  staff  96 Sep 18 15:03 ..


In [3]:
codigo_script = '''
print("Hola. Este script se ejecutó desde la terminal, no desde el notebook.")
'''

with open("bloque1_demo/hola_finanzas.py", "w") as f:
    f.write(codigo_script)

!python bloque1_demo/hola_finanzas.py

Hola. Este script se ejecutó desde la terminal, no desde el notebook.


### 1.3 Extensiones esenciales de VS Code

Una instalación mínima de VS Code para trabajo de datos necesita tres extensiones:

| Extensión | Para qué sirve |
|---|---|
| **Python** (Microsoft) | Autocompletado, selección de intérprete/entorno virtual, depuración |
| **Jupyter** (Microsoft) | Ejecutar y editar archivos `.ipynb` directamente dentro de VS Code |
| **GitLens** | Ver el historial de Git línea por línea, quién cambió qué y cuándo, directamente en el editor |

### 1.4 Notebooks (`.ipynb`) vs. scripts (`.py`): ¿cuándo usar cada uno?

| | Notebook (`.ipynb`) | Script (`.py`) |
|---|---|---|
| Mejor para | Explorar datos, prototipar, comunicar resultados con gráficos | Código que se va a reutilizar, probar y poner en producción |
| Orden de ejecución | Puede ejecutarse fuera de orden (fuente común de errores) | Siempre de arriba hacia abajo, predecible |
| Control de versiones con Git | El archivo mezcla código y salidas (imágenes, tablas) — los `diff` son difíciles de leer | `diff` limpio, línea por línea |
| Pruebas automatizadas (Bloque 5) | Difícil de probar con `pytest` | Se prueba de forma natural |

**Regla práctica:** explora y valida ideas en un notebook; en cuanto una función funciona y la vas a reutilizar, muévela a un archivo `.py` dentro de `src/` (lo practicamos en el Bloque 5).

**Ejercicio 1 (para tu propia terminal):** crea una carpeta `mi_primer_proyecto`, entra en ella con `cd`, crea un archivo `resumen.py` que imprima tu nombre y la fecha de hoy, y ejecútalo con `python resumen.py`.

## Bloque 2: Ambientes virtuales y dependencias

### 2.1 ¿Por qué aislar dependencias?

Imagina que trabajas en dos proyectos al mismo tiempo: un modelo de riesgo de crédito que depende de `scikit-learn==1.2` y un proyecto nuevo de series de tiempo que necesita una versión más reciente de `numpy` incompatible con esa versión de `scikit-learn`. Si instalas todo en el mismo Python "global" de tu computador, en algún momento un `pip install` de un proyecto rompe al otro.

Un **ambiente virtual** es una copia aislada del intérprete de Python con sus propias librerías, independiente de cualquier otro proyecto. Además de evitar conflictos, resuelve un problema aún más importante en finanzas: la **reproducibilidad**. Si un comité de riesgo pide reproducir exactamente un resultado de hace seis meses, necesitas poder recrear el entorno exacto en el que se generó — no "la versión más reciente de todo".

### 2.2 `venv`: crear, activar y seleccionar el intérprete

El módulo `venv` viene incluido con Python. El flujo típico, **en tu terminal**, es:

```bash
python3 -m venv .venv          # crea el ambiente en la carpeta .venv
source .venv/bin/activate      # lo activa (macOS/Linux)
.venv\Scripts\activate         # lo activa (Windows)
deactivate                     # lo desactiva, vuelve al Python global
```

En VS Code, después de crear el ambiente, selecciona el intérprete correcto con `Cmd/Ctrl + Shift + P` → **"Python: Select Interpreter"** → eliges la ruta dentro de `.venv`. Así el editor, la terminal integrada y el kernel de Jupyter usan siempre el mismo ambiente.

Dentro de este notebook no podemos "activar" un ambiente de forma persistente (cada celda con `!` es una terminal nueva), pero sí podemos crear uno real y usar directamente su `pip`, sin activarlo, para ver exactamente lo que ocurre por dentro.

In [4]:
!python3 -m venv bloque1_demo/.venv_demo
!ls bloque1_demo/.venv_demo

bin        include    lib        pyvenv.cfg


La carpeta contiene `bin/` (los ejecutables, incluido `python` y `pip` propios de este ambiente), `lib/` (donde se instalan las librerías) y `pyvenv.cfg` (la configuración del ambiente). Si alguna de estas piezas falta —algo que puede pasar si el ambiente se copió mal entre computadores o se interrumpió su creación— el ambiente queda inutilizable y hay que recrearlo desde cero.

### 2.3 `pip`: instalar y congelar dependencias

In [5]:
# Instalamos una librería directamente con el pip DE ESE ambiente (sin activarlo)
!bloque1_demo/.venv_demo/bin/pip install --quiet --disable-pip-version-check requests

# pip freeze imprime, en formato "paquete==version", todo lo instalado
!bloque1_demo/.venv_demo/bin/pip freeze > bloque1_demo/requirements.txt
!cat bloque1_demo/requirements.txt

certifi==2026.7.22
charset-normalizer==3.5.1
idna==3.20
requests==2.34.2
urllib3==2.8.0


Ese archivo `requirements.txt` es el contrato de reproducibilidad del proyecto: cualquier persona (o cualquier servidor) puede recrear exactamente el mismo ambiente con:

```bash
pip install -r requirements.txt
```

### 2.4 Alternativas: `conda` y `uv` (mención breve)

No las vamos a practicar en este cuaderno, pero es útil saber que existen y cuándo se prefieren:

| Herramienta | Cuándo se usa en la práctica |
|---|---|
| `venv` + `pip` | Estándar de la librería, suficiente para la mayoría de proyectos de Python puro |
| `conda` | Cuando el proyecto depende de librerías con componentes en C/Fortran difíciles de compilar (ej. paquetes científicos pesados, GPU) |
| `uv` | Alternativa moderna y mucho más rápida a `pip`/`venv`, cada vez más común en equipos nuevos |

**Ejercicio 2:** crea un ambiente virtual para un proyecto nuevo, actívalo en tu terminal, instala `pandas`, genera su `requirements.txt` y ábrelo para confirmar que aparece la versión instalada.

## Bloque 3: Git local

### 3.1 ¿Qué es el control de versiones?

Git guarda el historial completo de los cambios de un proyecto: quién cambió qué, cuándo y por qué. A diferencia de guardar copias como `analisis_v1.py`, `analisis_v2_final.py`, `analisis_v2_final_DEFINITIVO.py` (todos hemos hecho esto), Git guarda **un solo archivo** con **todo su historial** navegable.

De aquí en adelante vamos a construir, paso a paso y en vivo, un proyecto de ejemplo — un pipeline de precios de acciones — que iremos ampliando a lo largo de los bloques 3, 4 y 5.

In [6]:
import tempfile
import pathlib

demo_dir = pathlib.Path(tempfile.mkdtemp(prefix="proyecto_precios_acciones_"))
%cd {demo_dir}
print(f"Proyecto de ejemplo creado en: {demo_dir}")

/private/var/folders/tf/x2jqms8n4wn8gfljd87ypbl40000gn/T/proyecto_precios_acciones_oe773k5z
Proyecto de ejemplo creado en: /var/folders/tf/x2jqms8n4wn8gfljd87ypbl40000gn/T/proyecto_precios_acciones_oe773k5z


In [7]:
!git init -q
!git status

On branch main

No commits yet

nothing to commit (create/copy files and use "git add" to track)


`git status` es el comando que más vas a usar: siempre te dice en qué estado está tu proyecto. Ahora mismo dice "no commits yet" porque el repositorio está vacío. Creemos un primer archivo.

In [8]:
readme = '''# Proyecto Precios Acciones

Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones.
'''

with open("README.md", "w") as f:
    f.write(readme)

!git status

On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.md

nothing added to commit but untracked files present (use "git add" to track)


`README.md` aparece como **untracked** (Git lo ve, pero todavía no lo está siguiendo). El flujo básico es: `add` (lo prepara para el commit) → `commit` (lo guarda en el historial, con un mensaje).

In [9]:
!git add README.md
!git status

On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   README.md



In [10]:
!git commit -q -m "Agrega README inicial del proyecto"
!git log --oneline

8823976 (HEAD -> main) Agrega README inicial del proyecto


| Comando | Qué hace |
|---|---|
| `git init` | Convierte la carpeta actual en un repositorio Git |
| `git status` | Muestra qué archivos cambiaron, cuáles están preparados (*staged*) y cuáles no |
| `git add <archivo>` | Prepara un archivo para el próximo commit |
| `git commit -m "mensaje"` | Guarda una "fotografía" permanente de los archivos preparados |
| `git log` | Muestra el historial de commits |
| `git diff` | Muestra línea por línea qué cambió y aún no se ha preparado |

Modifiquemos el README y veamos `git diff` en acción, antes de confirmar el cambio con un nuevo commit.

In [11]:
with open("README.md", "a") as f:
    f.write("\n## Fuente de datos\n\nYahoo Finance, vía la librería `yfinance`.\n")

!git diff

diff --git a/README.md b/README.md
index 0ef5d34..554759b 100644
--- a/README.md
+++ b/README.md
@@ -1,3 +1,7 @@
 # Proyecto Precios Acciones
 
 Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones.
+
+## Fuente de datos
+
+Yahoo Finance, vía la librería `yfinance`.


In [12]:
!git add README.md
!git commit -q -m "Documenta la fuente de datos en el README"
!git log --oneline

251a066 (HEAD -> main) Documenta la fuente de datos en el README
8823976 Agrega README inicial del proyecto


### 3.2 `.gitignore`: lo que Git nunca debe guardar

No todo debe entrar al historial de Git: datos pesados, credenciales, o archivos temporales que genera el propio entorno. Simulemos algunos de esos archivos "peligrosos" y veamos cómo aparecen antes de tener un `.gitignore`.

In [13]:
import pathlib

pathlib.Path("data").mkdir(exist_ok=True)
pathlib.Path("data/precios_crudos.csv").write_text("fecha,precio\n2026-01-01,150.2\n")
pathlib.Path(".env").write_text("API_KEY=super-secreta-no-deberia-verse\n")
pathlib.Path("__pycache__").mkdir(exist_ok=True)
pathlib.Path("__pycache__/analisis.cpython-311.pyc").write_text("binario simulado")

!git status

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.env
	__pycache__/
	data/

nothing added to commit but untracked files present (use "git add" to track)


Los cuatro elementos aparecen como *untracked* — si alguien corriera `git add .` sin pensarlo, la clave de API y los datos crudos quedarían en el historial de Git **para siempre** (incluso si luego se borran, seguirían en commits antiguos). Un `.gitignore` evita justamente eso.

In [14]:
gitignore = '''# Datos
data/

# Credenciales y variables de entorno
.env

# Ambientes virtuales
venv/
.venv/

# Cachés de Python
__pycache__/

# Checkpoints de notebooks de Jupyter
.ipynb_checkpoints/
'''

with open(".gitignore", "w") as f:
    f.write(gitignore)

!git status

On branch main
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore

nothing added to commit but untracked files present (use "git add" to track)


Ahora `data/`, `.env` y `__pycache__/` desaparecieron de la lista de cambios — solo aparece `.gitignore`, que sí queremos versionar (para que el resto del equipo tenga las mismas reglas).

### 3.3 Mensajes de commit útiles

| Mensaje | Problema |
|---|---|
| `cambios` | No dice qué cambió ni por qué |
| `fix` | ¿Qué se arregló? |
| `asdasd` | — |
| **`Agrega validación de fechas faltantes en la serie de precios`** | Claro, específico, en modo imperativo — se entiende sin abrir el commit |

Una buena convención (no obligatoria, pero muy usada): **verbo en presente/imperativo + qué cambió**. Evita mensajes genéricos como "update" o "cambios varios".

In [15]:
!git add .gitignore
!git commit -q -m "Agrega .gitignore para datos, credenciales y cachés"
!git log --oneline

6dc6f25 (HEAD -> main) Agrega .gitignore para datos, credenciales y cachés
251a066 Documenta la fuente de datos en el README
8823976 Agrega README inicial del proyecto


### 3.4 Deshacer cambios: `restore` y `revert`

Dos escenarios distintos, dos comandos distintos:

- **`git restore <archivo>`**: descarta cambios que *todavía no se han confirmado* (no hay commit de por medio). Es "deshacer" en el sentido normal.
- **`git revert <commit>`**: crea un **nuevo commit** que deshace los cambios de un commit anterior, sin borrar el historial. Es la forma segura de deshacer algo que ya se compartió con el equipo.

Probemos primero `restore`: dañamos el README sin confirmar el cambio, y lo recuperamos.

In [16]:
with open("README.md", "a") as f:
    f.write("\n\nESTO FUE UN ERROR QUE NO QUERIA GUARDAR\n")

print("--- antes de restore ---")
print(pathlib.Path("README.md").read_text()[-60:])

!git restore README.md

print("\n--- después de restore ---")
print(pathlib.Path("README.md").read_text()[-60:])

--- antes de restore ---
rería `yfinance`.


ESTO FUE UN ERROR QUE NO QUERIA GUARDAR




--- después de restore ---
Fuente de datos

Yahoo Finance, vía la librería `yfinance`.



Ahora probemos `revert`: hacemos un commit "malo" a propósito, y lo deshacemos sin borrar el historial.

In [17]:
with open("README.md", "a") as f:
    f.write("\n## Sección que en realidad no queríamos\n")

!git add README.md
!git commit -q -m "Agrega seccion que no deberia estar"
!git log --oneline

52260cd (HEAD -> main) Agrega seccion que no deberia estar
6dc6f25 Agrega .gitignore para datos, credenciales y cachés
251a066 Documenta la fuente de datos en el README
8823976 Agrega README inicial del proyecto


In [18]:
!git revert --no-edit HEAD
!git log --oneline

[main 78122c2] Revert "Agrega seccion que no deberia estar"
 Date: Fri Sep 18 15:04:00 2026 -0500
 1 file changed, 2 deletions(-)


78122c2 (HEAD -> main) Revert "Agrega seccion que no deberia estar"
52260cd Agrega seccion que no deberia estar
6dc6f25 Agrega .gitignore para datos, credenciales y cachés
251a066 Documenta la fuente de datos en el README
8823976 Agrega README inicial del proyecto


Fíjate que `git revert` **no borró** el commit problemático: agregó uno nuevo que deshace sus cambios. El historial completo queda visible — exactamente el tipo de trazabilidad que un equipo financiero necesita poder mostrar.

**Ejercicio 3:** en tu propio repositorio, haz tres commits pequeños con mensajes descriptivos (por ejemplo, tres pasos de limpieza de un dataset), y luego deshaz el último con `git revert`. Verifica con `git log --oneline` que los tres commits originales siguen visibles.

### 3.5 Antes del Bloque 4: crea tu cuenta de GitHub (con GitHub Education)

Todo lo que hicimos en este bloque fue **local**: no necesitaste ninguna cuenta. Pero el Bloque 4 sí requiere una cuenta en GitHub — y si eres estudiante o docente, conviene tramitarla como cuenta educativa desde el principio, porque la verificación puede tardar y los beneficios (repositorios privados ilimitados, minutos extra de GitHub Actions para el Bloque 6, GitHub Copilot gratis, entre otros) se activan sobre la cuenta que ya tengas creada.

| Paso | Qué hacer |
|---|---|
| 1. Crear la cuenta base | Entra a [github.com/signup](https://github.com/signup) y regístrate. Si tienes correo institucional (`@tuuniversidad.edu.co`), úsalo agiliza la verificación del paso 3. Si no, puedes agregarlo después desde *Settings → Emails*. |
| 2. Solicitar el paquete educativo | Con sesión iniciada, ve a [education.github.com](https://education.github.com/) y elige **"Get student benefits"** (si eres estudiante) o **"Get teacher benefits"** (si eres docente). |
| 3. Verificar tu afiliación académica | GitHub te pide país, institución y fecha esperada de grado. Puede verificarte de dos formas: automáticamente si usas un correo `.edu`/institucional reconocido, o pidiéndote subir una foto de tu carnet estudiantil o un certificado de matrícula vigente si no tienes ese correo. |
| 4. Esperar la aprobación | La verificación automática por correo institucional suele ser casi inmediata; la revisión manual de documentos puede tardar unos días. GitHub te avisa por correo y con una notificación en la plataforma. |
| 5. Confirmar los beneficios activos | Una vez aprobado, entra a [education.github.com/pack](https://education.github.com/pack) para ver el catálogo completo (GitHub Copilot, más minutos de Actions, créditos en otras herramientas como JetBrains o Canva, entre otros). |

> Si tu institución no tiene correo `.edu` reconocido por GitHub, no pasa nada: la verificación manual con carnet o certificado de matrícula funciona igual, solo toma un poco más de tiempo. En cualquier caso, con la cuenta base del paso 1 ya puedes hacer todo lo del Bloque 4 — el paquete educativo es un beneficio adicional, no un requisito.

## Bloque 4: GitHub y colaboración

### 4.1 De local a remoto: GitHub

Todo lo del Bloque 3 vivía solo en tu computador. GitHub es un servidor donde alojar una copia remota del repositorio, para poder compartirlo y colaborar. Estos comandos requieren una cuenta en [github.com](https://github.com) y **no los ejecutamos aquí** (crearían un repositorio real) — practícalos en tu propia terminal:

```bash
# 1. Crea un repositorio vacío en github.com (botón "New repository")

# 2. Conecta tu repositorio local con el remoto
git remote add origin https://github.com/tu-usuario/tu-repositorio.git

# 3. Sube tu historial local por primera vez
git push -u origin main

# 4. En adelante, para subir nuevos commits
git push

# 5. Para traer cambios que otras personas subieron
git pull

# 6. Para obtener una copia completa de un repositorio existente
git clone https://github.com/tu-usuario/tu-repositorio.git
```

### 4.2 Ramas, Pull Requests y revisión de código

Una **rama** (*branch*) es una línea de desarrollo independiente: te permite trabajar en una funcionalidad nueva sin afectar la rama principal (`main`) hasta que esté lista. Un **Pull Request (PR)** es la propuesta, en GitHub, de fusionar los cambios de una rama hacia otra — y el espacio natural donde el resto del equipo revisa el código antes de aceptarlo.

Creemos dos ramas locales para simular ese flujo (`git branch`/`git checkout -b` sí son completamente locales, no requieren GitHub).

In [19]:
!git branch -M main
!git checkout -qb agrega-modelo-base
!git branch

* agrega-modelo-base
  main


**Simulación en parejas:** en el taller, cada persona trabaja en su propia rama sobre el mismo repositorio remoto, abre un Pull Request al terminar, y su pareja lo revisa en GitHub dejando al menos un comentario antes de aprobarlo (botón *"Review changes" → "Approve"* o *"Request changes"*). Es exactamente el flujo que usarás en un equipo real.

### 4.3 README, licencia y tu perfil de GitHub como portafolio

Tu perfil de GitHub es, en la práctica, parte de tu hoja de vida para roles de datos. Dos elementos mínimos por repositorio:

- **`README.md`**: qué hace el proyecto, cómo instalarlo (`pip install -r requirements.txt`) y cómo ejecutarlo. Es lo primero que ve cualquier reclutador o colega.
- **Licencia**: define qué puede hacer otra persona con tu código. Para proyectos de portafolio, `MIT` (permisiva, casi sin restricciones) es la opción más común; GitHub la puede generar automáticamente al crear el repositorio.

### 4.4 Resolver un conflicto de *merge*

Un conflicto ocurre cuando dos ramas modificaron **la misma línea** de un archivo de formas distintas, y Git no puede decidir cuál versión conservar. Vamos a provocar uno a propósito y resolverlo.

In [20]:
# Modificamos la misma línea del README en la rama "agrega-modelo-base"
contenido = pathlib.Path("README.md").read_text()
contenido = contenido.replace(
    "Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones.",
    "Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones usando un modelo LSTM.",
)
pathlib.Path("README.md").write_text(contenido)

!git add README.md
!git commit -q -m "Menciona el modelo LSTM en el README"

# Volvemos a main y creamos una SEGUNDA rama que edita la MISMA línea, de otra forma
!git checkout -q main
!git checkout -qb agrega-modelo-alternativo

In [21]:
contenido = pathlib.Path("README.md").read_text()
contenido = contenido.replace(
    "Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones.",
    "Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones usando un modelo MLP.",
)
pathlib.Path("README.md").write_text(contenido)

!git add README.md
!git commit -q -m "Menciona el modelo MLP en el README"

In [22]:
# Fusionamos la primera rama en main: esta funciona sin problema
!git checkout -q main
!git merge --no-edit agrega-modelo-base

# Ahora intentamos fusionar la segunda rama, que cambió la MISMA línea: esto sí genera conflicto
!git merge --no-edit agrega-modelo-alternativo

Updating 78122c2..c4dbf60
Fast-forward
 README.md | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


Auto-merging README.md
CONFLICT (content): Merge conflict in README.md
Automatic merge failed; fix conflicts and then commit the result.


Git avisa el conflicto y marca el archivo directamente. Así se ve por dentro (en VS Code, estos mismos marcadores aparecen resaltados con botones para elegir una versión, ambas, o escribir una nueva):

In [23]:
print(pathlib.Path("README.md").read_text())

# Proyecto Precios Acciones

<<<<<<< HEAD
Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones usando un modelo LSTM.
Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones usando un modelo MLP.
>>>>>>> agrega-modelo-alternativo

## Fuente de datos

Yahoo Finance, vía la librería `yfinance`.



`<<<<<<< HEAD` marca el inicio de la versión de la rama en la que estás parado; `=======` separa las dos versiones; `>>>>>>> agrega-modelo-alternativo` marca el final de la versión que se intentó fusionar. Resolver el conflicto es decidir qué texto final queremos — aquí optamos por conservar los dos modelos, en vez de elegir uno solo.

In [24]:
resolucion = '''# Proyecto Precios Acciones

Pipeline de ejemplo para descargar, limpiar y modelar precios de acciones usando un modelo MLP y un modelo LSTM.

## Fuente de datos

Yahoo Finance, vía la librería `yfinance`.

## Sección que en realidad no queríamos
'''

pathlib.Path("README.md").write_text(resolucion)

!git add README.md
!git commit -q -m "Resuelve conflicto: conserva ambos modelos en el README"
!git log --oneline --graph --all

*   4d15601 (HEAD -> main) Resuelve conflicto: conserva ambos modelos en el README
|\  
| * 9d0e844 (agrega-modelo-alternativo) Menciona el modelo MLP en el README
* | c4dbf60 (agrega-modelo-base) Menciona el modelo LSTM en el README
|/  
* 78122c2 Revert "Agrega seccion que no deberia estar"
* 52260cd Agrega seccion que no deberia estar
* 6dc6f25 Agrega .gitignore para datos, credenciales y cachés
* 251a066 Documenta la fuente de datos en el README
* 8823976 Agrega README inicial del proyecto


El `--graph` deja ver visualmente cómo las dos ramas divergieron y volvieron a unirse en `main`. Esto es exactamente lo que ocurre, a mayor escala, al fusionar un Pull Request con conflictos en GitHub — la diferencia es que ahí la resolución se hace desde la interfaz web o desde VS Code, no editando el archivo a mano en Python como hicimos aquí por claridad.

**Ejercicio 4:** con un compañero (o tú mismo, usando dos ramas), edita la misma línea de un archivo en dos ramas distintas, fusiona ambas en `main` y resuelve el conflicto conservando información de las dos versiones.

## Bloque 5: Estructura de proyecto y buenas prácticas

### 5.1 Estructura estándar de un proyecto de datos

Un proyecto reproducible sigue una convención de carpetas que cualquier persona del equipo reconoce de inmediato:

```
proyecto_precios_acciones/
├── data/              # datos crudos y procesados (NO se versiona en Git)
├── notebooks/         # exploración y prototipado
├── src/               # código reutilizable, en funciones
├── tests/             # pruebas automatizadas de ese código
├── requirements.txt   # dependencias exactas (Bloque 2)
├── .gitignore         # qué no versionar (Bloque 3)
└── README.md          # qué es el proyecto y cómo ejecutarlo
```

Esta convención no es un invento propio: sigue de cerca a **[Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/)**, una plantilla de proyecto muy usada en la industria que genera automáticamente esta estructura (no la vamos a instalar aquí, pero vale la pena conocerla para proyectos nuevos).

Construyamos esta estructura dentro de nuestro proyecto de ejemplo.

In [25]:
for carpeta in ["notebooks", "src", "tests"]:
    pathlib.Path(carpeta).mkdir(exist_ok=True)

pathlib.Path("src/__init__.py").touch()

!find . -not -path "./.git*" -not -path "./bloque1_demo*" | sort

.
./.env
./README.md
./__pycache__
./__pycache__/analisis.cpython-311.pyc
./data
./data/precios_crudos.csv
./notebooks
./src
./src/__init__.py
./tests


### 5.2 De notebook a `src/`: convertir código en funciones

En un notebook es común escribir cálculos "sueltos", pegados directamente en una celda. El primer paso hacia un proyecto reproducible es envolver ese cálculo en una **función con nombre**, guardarla en `src/`, y luego importarla — en vez de copiar y pegar el mismo código en cada notebook nuevo.

In [26]:
# Así se ve el cálculo "suelto", estilo notebook exploratorio
precios = [150.2, 152.8, 149.5, 155.1]
retorno_promedio = sum((precios[i] / precios[i - 1] - 1) for i in range(1, len(precios))) / (len(precios) - 1)
print(f"Retorno diario promedio: {retorno_promedio:.4%}")

Retorno diario promedio: 1.1057%


In [27]:
codigo_modulo = '''# Funciones de procesamiento de precios para el proyecto


def retorno_promedio(precios):
    # Calcula el retorno porcentual promedio entre precios consecutivos
    retornos = [precios[i] / precios[i - 1] - 1 for i in range(1, len(precios))]
    return sum(retornos) / len(retornos)
'''

with open("src/procesamiento.py", "w") as f:
    f.write(codigo_modulo)

!cat src/procesamiento.py

# Funciones de procesamiento de precios para el proyecto


def retorno_promedio(precios):
    # Calcula el retorno porcentual promedio entre precios consecutivos
    retornos = [precios[i] / precios[i - 1] - 1 for i in range(1, len(precios))]
    return sum(retornos) / len(retornos)


In [28]:
import sys
sys.path.insert(0, str(demo_dir))

from src.procesamiento import retorno_promedio

print(f"Retorno diario promedio (desde src/): {retorno_promedio(precios):.4%}")

Retorno diario promedio (desde src/): 1.1057%


Mismo resultado, pero ahora la lógica vive en un solo lugar (`src/procesamiento.py`), es importable desde cualquier notebook del proyecto, y —como veremos enseguida— se puede probar automáticamente.

### 5.3 Variables de entorno y `.env`

Nunca se escribe una clave de API directamente en el código (recuerda el Bloque 3: si el archivo se versiona, la clave queda en el historial de Git para siempre, aunque se borre después). La práctica estándar es guardarla en un archivo `.env` — que el `.gitignore` del Bloque 3 ya excluye — y leerla desde el código.

In [29]:
pathlib.Path(".env").write_text("API_KEY_MERCADO=clave-de-ejemplo-nunca-subir-esto\n")

from dotenv import load_dotenv
import os

load_dotenv()  # lee el archivo .env y carga sus variables al entorno del proceso
clave = os.environ.get("API_KEY_MERCADO")

print(f"Clave cargada desde .env: {clave}")
print()
!git status --short .env

Clave cargada desde .env: clave-de-ejemplo-nunca-subir-esto



`git status --short .env` no muestra ninguna línea: el `.gitignore` que creamos en el Bloque 3 ya cubre este archivo. La clave existe y es usable dentro del proyecto, pero nunca viaja al repositorio.

### 5.4 Formateo con `black` y *linting* con `ruff`

Un **formateador** (`black`) reescribe automáticamente tu código para que siga un estilo consistente (espacios, comillas, saltos de línea) — elimina las discusiones de estilo en las revisiones de código. Un ***linter*** (`ruff`) va más allá: detecta errores probables (variables sin usar, importaciones que sobran, comparaciones sospechosas) antes de que lleguen a producción.

Escribamos un archivo con formato descuidado a propósito.

In [30]:
codigo_desordenado = '''import os
import sys
def calcular_variacion(precio_inicial,precio_final):
    variacion = (precio_final-precio_inicial)/precio_inicial
    return variacion
'''

with open("src/variacion.py", "w") as f:
    f.write(codigo_desordenado)

print("--- antes de black ---")
print(pathlib.Path("src/variacion.py").read_text())

--- antes de black ---
import os
import sys
def calcular_variacion(precio_inicial,precio_final):
    variacion = (precio_final-precio_inicial)/precio_inicial
    return variacion



In [31]:
!black -q src/variacion.py

print("--- después de black ---")
print(pathlib.Path("src/variacion.py").read_text())

--- después de black ---
import os
import sys


def calcular_variacion(precio_inicial, precio_final):
    variacion = (precio_final - precio_inicial) / precio_inicial
    return variacion



`black` corrigió el espaciado alrededor de los operadores y los parámetros — sin cambiar la lógica. Pero el archivo todavía tiene un problema que el formateador no detecta: importa `os` y `sys` y nunca los usa. Ahí es donde entra `ruff`.

In [32]:
!ruff check src/variacion.py

F401 [*] `os` imported but unused
 --> src/variacion.py:1:8
  |
1 | import os
  |        ^^
2 | import sys
  |
help: Remove unused import: `os`
  |
  - import os
1 | import sys
  |

F401 [*] `sys` imported but unused
 --> src/variacion.py:2:8
  |
1 | import os
2 | import sys
  |        ^^^
help: Remove unused import: `sys`
  |
1 | import os
  - import sys
2 |
  |

Found 2 errors.
[*] 2 fixable with the `--fix` option.


In [33]:
# Corregimos el problema que reportó ruff: quitamos las importaciones sin usar
codigo_corregido = pathlib.Path("src/variacion.py").read_text().replace("import os\nimport sys\n", "")
pathlib.Path("src/variacion.py").write_text(codigo_corregido)

!ruff check src/variacion.py

All checks passed!


`ruff check` ya no reporta nada: es la señal (usada también por herramientas de integración continua, Bloque 6) de que el archivo pasa el control de calidad.

### 5.5 Un test mínimo con `pytest`

Una prueba automatizada verifica que una función siga funcionando como se espera, incluso después de que alguien más la modifique meses después. `pytest` reconoce automáticamente cualquier archivo `test_*.py` y cualquier función `test_*` dentro de él.

In [34]:
codigo_test = '''import sys
import pathlib

sys.path.insert(0, str(pathlib.Path(__file__).resolve().parents[1]))

from src.procesamiento import retorno_promedio


def test_retorno_promedio_con_precios_constantes():
    # Si el precio nunca cambia, el retorno promedio debe ser exactamente 0
    assert retorno_promedio([100, 100, 100]) == 0


def test_retorno_promedio_con_precios_crecientes():
    # Con precios que suben 10% cada día, el promedio debe ser ~10%
    assert abs(retorno_promedio([100, 110, 121]) - 0.10) < 1e-9
'''

with open("tests/test_procesamiento.py", "w") as f:
    f.write(codigo_test)

!pytest -q

..                                                                       [100%]
2 passed in 0.00s


`2 passed` confirma que la función se comporta como esperamos en ambos escenarios. Si en el futuro alguien cambia `retorno_promedio` y rompe alguno de los dos casos, `pytest` lo va a marcar en rojo de inmediato — antes de que ese error llegue a producción.

**Ejercicio 5:** agrega a `src/procesamiento.py` una función `retorno_total(precios)` que calcule el retorno acumulado entre el primer y el último precio, y escribe al menos un caso de prueba para ella en `tests/test_procesamiento.py`.

## Bloque 6: Panorama MLOps y cierre

### 6.1 El ciclo de vida de un modelo

DataOps cubre el proyecto en general; **MLOps** es su aplicación específica al ciclo de vida de un modelo de *machine learning*:

```
datos → entrenamiento → versionado → despliegue → monitoreo
  ↑                                                    │
  └────────────────── reentrenamiento ←────────────────┘
```

Cada etapa tiene una pregunta que responde:

| Etapa | Pregunta que responde | Ejemplo financiero |
|---|---|---|
| Datos | ¿De dónde vienen y son confiables? | Precios de mercado, estados financieros, buró de crédito |
| Entrenamiento | ¿Qué modelo y con qué configuración? | Un modelo de score de crédito, entrenado con datos históricos de mora |
| Versionado | ¿Puedo reproducir exactamente esta versión del modelo? | Guardar el modelo, sus datos de entrenamiento y su código juntos |
| Despliegue | ¿Cómo llega el modelo a producción? | El modelo corre en el motor de aprobación de créditos en tiempo real |
| Monitoreo | ¿Sigue funcionando bien con datos nuevos? | Si el comportamiento de pago cambia (una crisis económica), el modelo entrenado antes puede perder precisión — esto se llama *drift* |

El ciclo se cierra con el monitoreo: cuando el desempeño se degrada, hay que reentrenar con datos más recientes y volver a empezar.

### 6.2 MLflow: registro de experimentos

Entrenar un modelo casi nunca funciona a la primera. En el camino se prueban decenas de configuraciones (variables distintas, ventanas distintas, arquitecturas distintas, como en nuestro cuaderno de MLP/LSTM). **MLflow** registra automáticamente cada intento —sus parámetros, sus métricas de desempeño— para poder comparar y volver atrás sin depender de la memoria o de notas sueltas.

Registremos, dentro de nuestro proyecto de ejemplo, dos "experimentos" simulados con distintas configuraciones.

In [35]:
import os
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"  # silencia una sugerencia de MLflow dirigida a agentes de IA, no relevante aquí

import mlflow

mlflow.set_tracking_uri(f"sqlite:///{demo_dir}/mlflow.db")
mlflow.set_experiment("prediccion_precios_accion")

configuraciones = [
    {"ventana_dias": 15, "unidades_lstm": 30, "rmse_test": 5.9},
    {"ventana_dias": 30, "unidades_lstm": 50, "rmse_test": 5.08},
]

for config in configuraciones:
    with mlflow.start_run():
        mlflow.log_param("ventana_dias", config["ventana_dias"])
        mlflow.log_param("unidades_lstm", config["unidades_lstm"])
        mlflow.log_metric("rmse_test", config["rmse_test"])

print("Dos corridas registradas en MLflow.")

2026/09/18 15:04:08 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/09/18 15:04:08 INFO mlflow.store.db.utils: Updating database tables


2026/09/18 15:04:08 INFO mlflow.tracking.fluent: Experiment with name 'prediccion_precios_accion' does not exist. Creating a new experiment.


Dos corridas registradas en MLflow.


In [36]:
corridas = mlflow.search_runs(experiment_names=["prediccion_precios_accion"])
corridas[["run_id", "params.ventana_dias", "params.unidades_lstm", "metrics.rmse_test"]]

,run_id,params.ventana_dias,params.unidades_lstm,metrics.rmse_test
0,09c16afb95e94abf853b6986c5a5ddcb,30,50,5.08
1,a9920481ab8140f3a1782dc44e2a7f5a,15,30,5.90


`mlflow.search_runs` confirma que ambas corridas quedaron registradas con sus parámetros y su métrica — sin haber anotado nada a mano. Usamos una base de datos SQLite local (`mlflow.db`) como backend de registro: desde la versión 3, MLflow dejó en modo mantenimiento el backend de solo archivos que usaban versiones anteriores. En un proyecto real, después de correr esto en tu terminal (**no** dentro de un notebook, porque levanta un servidor), con:

```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db
```

se abre en `http://localhost:5000` una interfaz visual para comparar todas las corridas, ordenarlas por métrica y ver cuál configuración fue la mejor.

### 6.3 DVC: versionado de datos

Git funciona muy bien para código, pero **no** para archivos de datos grandes (un CSV de varios GB, por ejemplo): los guarda completos en cada versión y el repositorio crece sin control. **DVC (Data Version Control)** resuelve esto: guarda en Git solo un archivo liviano de referencia (un *hash*), mientras que los datos reales viven en almacenamiento externo (S3, Google Drive, un servidor propio).

No lo vamos a instalar ni ejecutar aquí, pero así se ve el flujo típico, en tu terminal:

```bash
dvc init
dvc add data/precios_crudos.csv     # crea data/precios_crudos.csv.dvc (esto sí se versiona con git)
git add data/precios_crudos.csv.dvc .gitignore
git commit -m "Versiona referencia a los datos crudos con DVC"
dvc push                            # sube los datos reales al almacenamiento remoto configurado
```

### 6.4 Docker: reproducibilidad más allá de Python

Un ambiente virtual (Bloque 2) aísla las librerías de Python, pero no aísla la versión del sistema operativo, ni servicios externos como una base de datos. **Docker** empaqueta la aplicación completa —código, dependencias de Python, y hasta el sistema operativo base— en un **contenedor** que corre exactamente igual en tu computador, en el de un colega, o en un servidor en la nube. Es la respuesta definitiva al "en mi máquina sí funciona".

No vamos a instalar Docker en este cuaderno, pero así se ve la receta (`Dockerfile`) de un proyecto como el nuestro:

```dockerfile
FROM python:3.11-slim

WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt

COPY src/ src/
CMD ["python", "src/procesamiento.py"]
```

Cada línea es un paso reproducible: parte de una imagen base de Python, instala las dependencias exactas del `requirements.txt`, copia el código, y define qué ejecutar al arrancar el contenedor.

### 6.5 GitHub Actions: pruebas automáticas en cada cambio

Ya escribimos pruebas con `pytest` (Bloque 5) y las corrimos manualmente. **GitHub Actions** las corre automáticamente cada vez que alguien sube un cambio o abre un Pull Request — sin que nadie tenga que acordarse de hacerlo. Es la pieza que cierra el ciclo de **integración continua (CI)**: ningún cambio se fusiona a `main` sin que las pruebas pasen.

Creemos, dentro de nuestro proyecto de ejemplo, el archivo de configuración real que activa esto.

In [37]:
workflow_ci = '''name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependencias
        run: pip install -r requirements.txt

      - name: Ejecutar pruebas
        run: pytest -q
'''

pathlib.Path(".github/workflows").mkdir(parents=True, exist_ok=True)
pathlib.Path(".github/workflows/ci.yml").write_text(workflow_ci)

!cat .github/workflows/ci.yml

name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Instalar dependencias
        run: pip install -r requirements.txt

      - name: Ejecutar pruebas
        run: pytest -q


Léelo de arriba hacia abajo: **`on`** define cuándo se dispara (cada `push` o Pull Request hacia `main`); **`jobs`** define qué máquina usar (`ubuntu-latest`, provista gratis por GitHub) y los pasos a ejecutar en orden — descargar el código (`checkout`), instalar Python, instalar dependencias, y correr `pytest`. Si `pytest` termina en error, GitHub marca el *check* en rojo y bloquea la fusión del Pull Request hasta que se corrija.

Este archivo, en un repositorio real, solo necesita estar en la ruta `.github/workflows/ci.yml` para que GitHub lo active automáticamente — no requiere ninguna configuración adicional en la web.

### 6.6 Cierre: tu entregable final

El taller completo (bloques 1 a 6) se cierra con un entregable único que reúne todo lo practicado. Publica un **repositorio público en GitHub** que contenga:

- [ ] `README.md` explicando qué hace el proyecto y cómo instalarlo/ejecutarlo.
- [ ] `requirements.txt` con las dependencias exactas (`pip freeze`).
- [ ] Código reutilizable dentro de `src/` (no solo notebooks sueltos).
- [ ] Al menos **un test** en `tests/`, ejecutable con `pytest`.
- [ ] Un *workflow* de GitHub Actions (`.github/workflows/ci.yml`) que corra ese test automáticamente — y que el *badge* de CI aparezca **en verde** en la página del repositorio.

Con eso tienes, en un solo repositorio, evidencia concreta de las seis competencias que cubrió este cuaderno: entorno reproducible, control de versiones, colaboración, buenas prácticas de código y una primera pieza real de automatización — exactamente lo que un equipo de datos financieros espera de alguien que se incorpora al equipo.